In [1]:
print("hello")

hello


In [2]:
import os
import sys
from pathlib import Path
 
from dotenv import load_dotenv, find_dotenv

In [3]:
import os
import sys
from pathlib import Path
 
from dotenv import load_dotenv, find_dotenv
 

_cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "multi_doc").is_dir()), _cwd)
os.chdir(PROJECT_ROOT)  
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))  
 
load_dotenv(find_dotenv(usecwd=True), override=True)
 
tracing = (os.getenv("LANGSMITH_TRACING") or os.getenv("LANGCHAIN_TRACING_V2") or "").lower()
print("Project root:      ", PROJECT_ROOT)
print("Tracing enabled:   ", tracing == "true")
print("LangSmith key:     ", bool(os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")))
print("LangSmith project: ", os.getenv("LANGSMITH_PROJECT") or os.getenv("LANGCHAIN_PROJECT") or "default")
print("Groq key:          ", bool(os.getenv("GROQ_API_KEY")))

Project root:       D:\RAG-System-LLM-Evaluation
Tracing enabled:    True
LangSmith key:      True
LangSmith project:  RAG_Chat_model
Groq key:           True


In [4]:
import pandas as pd
 
inputs = [
    "How many identical layers are stacked in the encoder of the base Transformer model?",
    "How many parallel attention heads does the base Transformer use, and what is the dimension of each head's keys and values?",
    "Why are the dot products scaled by 1/sqrt(d_k) in scaled dot-product attention?",
    "How does the Transformer give the model information about word order, since it has no recurrence or convolution?",
    "What BLEU score did the big Transformer achieve on the WMT 2014 English-to-German translation task?",
    "Which optimizer and learning-rate schedule were used to train the Transformer?",
    "What is the maximum path length between any two positions in a self-attention layer compared with a recurrent layer?",
    "What hardware was used to train the base Transformer models, and for how long?",
]
 
outputs = [
    "The encoder is a stack of N = 6 identical layers.",
    "The base model uses h = 8 parallel attention heads, and each head has keys and values of dimension d_k = d_v = 64 (d_model = 512 divided by 8).",
    "For large values of d_k the dot products grow large in magnitude, pushing the softmax into regions with extremely small gradients. Scaling by 1/sqrt(d_k) counteracts this.",
    "It adds positional encodings to the input embeddings at the bottom of the encoder and decoder stacks. The encodings are sine and cosine functions of different frequencies.",
    "The big Transformer achieved 28.4 BLEU, improving over the previous best results, including ensembles, by more than 2 BLEU.",
    "The Adam optimizer with beta1 = 0.9, beta2 = 0.98 and epsilon = 1e-9, with a learning rate that increases linearly for the first 4000 warmup steps and then decreases proportionally to the inverse square root of the step number.",
    "In a self-attention layer any two positions are connected by a constant, O(1), path length, whereas a recurrent layer needs O(n) sequential operations.",
    "They were trained on one machine with 8 NVIDIA P100 GPUs; each base model trained for 100,000 steps, about 12 hours.",
]
 
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]
df = pd.DataFrame(qa_pairs)
 
csv_path = PROJECT_ROOT / "data" / "goldens.csv"
df.to_csv(csv_path, index=False)
print(f"Saved {len(df)} golden pairs to {csv_path}")
df

Saved 8 golden pairs to D:\RAG-System-LLM-Evaluation\data\goldens.csv


,question,answer
0,How many identical layers are stacked in the e...,The encoder is a stack of N = 6 identical layers.
1,How many parallel attention heads does the bas...,The base model uses h = 8 parallel attention h...
2,Why are the dot products scaled by 1/sqrt(d_k)...,For large values of d_k the dot products grow ...
3,How does the Transformer give the model inform...,It adds positional encodings to the input embe...
4,What BLEU score did the big Transformer achiev...,"The big Transformer achieved 28.4 BLEU, improv..."
5,Which optimizer and learning-rate schedule wer...,"The Adam optimizer with beta1 = 0.9, beta2 = 0..."
6,What is the maximum path length between any tw...,In a self-attention layer any two positions ar...
7,What hardware was used to train the base Trans...,They were trained on one machine with 8 NVIDIA...


In [5]:
from langsmith import Client
 
client = Client()
dataset_name = "AttentionPaperGoldens"
 
if client.has_dataset(dataset_name=dataset_name):
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"Dataset '{dataset_name}' already exists, reusing it.")
else:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Question/answer pairs for the 'Attention Is All You Need' paper",
    )
    client.create_examples(
        inputs=[{"question": q} for q in inputs],
        outputs=[{"answer": a} for a in outputs],
        dataset_id=dataset.id,
    )
    print(f"Created dataset '{dataset_name}' with {len(inputs)} examples.")

Dataset 'AttentionPaperGoldens' already exists, reusing it.


In [8]:
from multi_doc.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc.src.document_chat.retrieval import ConversationalRAG
 
PAPER_PATH = PROJECT_ROOT / "data" / "NIPS-2017-attention-is-all-you-need-Paper.pdf"
 
 
class LocalFileAdapter:
    """Lets ChatIngestor read a file from disk (it expects .name and .getbuffer())."""
 
    def __init__(self, file_path):
        self.path = Path(file_path)
        self.name = self.path.name
 
    def getbuffer(self) -> bytes:
        return self.path.read_bytes()
 
 
_rag = None  
 
 
def get_rag() -> ConversationalRAG:
    """Index the paper once and return the ready-to-use RAG chain."""
    global _rag
    if _rag is not None:
        return _rag
 
    if not PAPER_PATH.exists():
        raise FileNotFoundError(f"Paper not found: {PAPER_PATH}")
 
    ingestor = ChatIngestor(
        temp_base=str(PROJECT_ROOT / "data"),
        faiss_base=str(PROJECT_ROOT / "faiss_index"),
        use_session_dirs=True,
    )
    ingestor.built_retriver([LocalFileAdapter(PAPER_PATH)])
 
    rag = ConversationalRAG(session_id=ingestor.session_id)
    rag.load_retriever_from_faiss(index_path=str(ingestor.faiss_dir))
    _rag = rag
    return _rag
 
 
def answer_attention_question(inputs: dict) -> dict:
    """LangSmith target function: {"question": ...} -> {"answer": ...}."""
    question = (inputs or {}).get("question", "").strip()
    if not question:
        return {"answer": "No question provided"}
 
    answer = get_rag().invoke(question, chat_history=[])
    return {"answer": answer}

In [9]:
test_input = {"question": inputs[0]}
result = answer_attention_question(test_input)
 
print("Question:", test_input["question"])
print("\nExpected:", outputs[0])
print("\nActual:  ", result["answer"])

{"timestamp": "2026-11-05T09:10:32.066849Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"keys": ["GOOGLE_API_KEY", "GROQ_API_KEY"], "timestamp": "2026-11-05T09:10:32.067883Z", "level": "info", "event": "API keys available"}
{"config_keys": ["embedding_model", "chunking", "vector_store", "retriever", "active_llm", "llm"], "timestamp": "2026-11-05T09:10:32.071411Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20261105_144032_e8888171", "temp_dir": "D:\\RAG-System-LLM-Evaluation\\data\\session_20261105_144032_e8888171", "faiss_dir": "D:\\RAG-System-LLM-Evaluation\\faiss_index\\session_20261105_144032_e8888171", "sessionized": true, "timestamp": "2026-11-05T09:10:32.073921Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "NIPS-2017-attention-is-all-you-need-Paper.pdf", "saved_as": "D:\\RAG-System-LLM-Evaluation\\data\\session_20261105_144032_e8888171\\nips-2017-attention-is-all-you-need-paper_d94cd3.pdf", "timestamp": "

Question: How many identical layers are stacked in the encoder of the base Transformer model?

Expected: The encoder is a stack of N = 6 identical layers.

Actual:   The base Transformer’s encoder stacks **6 identical layers** (N = 6) (nips‑2017‑attention‑is‑all‑you‑need‑paper_d94cd3.pdf, p. 2).


In [10]:
print("Testing all questions from the dataset:\n")
for i, (q, expected) in enumerate(zip(inputs, outputs), 1):
    result = answer_attention_question({"question": q})
    print(f"Q{i}: {q}")
    print(f"Expected: {expected}")
    print(f"Actual:   {result['answer']}\n")
    print("-" * 80 + "\n")

Testing all questions from the dataset:



HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20261105_144032_e8888171", "answer_preview": "The base Transformer\u2019s encoder stacks **6 identical layers** (N\u202f=\u202f6) (nips\u20112017\u2011attention\u2011is\u2011all\u2011you\u2011need\u2011paper_d94cd3.pdf, p.\u202f2).", "timestamp": "2026-11-05T09:11:14.507186Z", "level": "info", "event": "Chain invoked successfully"}


Q1: How many identical layers are stacked in the encoder of the base Transformer model?
Expected: The encoder is a stack of N = 6 identical layers.
Actual:   The base Transformer’s encoder stacks **6 identical layers** (N = 6) (nips‑2017‑attention‑is‑all‑you‑need‑paper_d94cd3.pdf, p. 2).

--------------------------------------------------------------------------------



HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20261105_144032_e8888171", "answer_preview": "The base Transformer uses **8 parallel attention heads**.  \nEach head has key and value dimensions of **64** (dk\u202f=\u202fdv\u202f=\u202f64). (nips\u20112017\u2011attention\u2011is\u2011a", "timestamp": "2026-11-05T09:11:15.472900Z", "level": "info", "event": "Chain invoked successfully"}


Q2: How many parallel attention heads does the base Transformer use, and what is the dimension of each head's keys and values?
Expected: The base model uses h = 8 parallel attention heads, and each head has keys and values of dimension d_k = d_v = 64 (d_model = 512 divided by 8).
Actual:   The base Transformer uses **8 parallel attention heads**.  
Each head has key and value dimensions of **64** (dk = dv = 64). (nips‑2017‑attention‑is‑all‑you‑need‑paper_d94cd3.pdf, p. 5; p. 9)

--------------------------------------------------------------------------------



HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20261105_144032_e8888171", "answer_preview": "The dot products are scaled by \\(1/\\sqrt{d_k}\\) to prevent them from becoming too large as the key/query dimension \\(d_k\\) grows. Large dot\u2011product va", "timestamp": "2026-11-05T09:11:16.405243Z", "level": "info", "event": "Chain invoked successfully"}


... (7 outputs left out)

HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20261105_144032_e8888171", "answer_preview": "The maximum path length between any two positions is:\n\n- **Self\u2011attention:** \\(O(1)\\) (constant) (nips\u20112017\u2011attention\u2011is\u2011all\u2011you\u2011need\u2011paper_d94cd3.pdf", "timestamp": "2026-11-05T09:12:05.923637Z", "level": "info", "event": "Chain invoked successfully"}
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
Retrying request to /openai/v1/chat/completions in 13.000000 seconds


Q7: What is the maximum path length between any two positions in a self-attention layer compared with a recurrent layer?
Expected: In a self-attention layer any two positions are connected by a constant, O(1), path length, whereas a recurrent layer needs O(n) sequential operations.
Actual:   The maximum path length between any two positions is:

- **Self‑attention:** \(O(1)\) (constant) (nips‑2017‑attention‑is‑all‑you‑need‑paper_d94cd3.pdf, p. 6)  
- **Recurrent:** \(O(n)\) (linear in the sequence length) (nips‑2017‑attention‑is‑all‑you‑need‑paper_d94cd3.pdf, p. 6)

--------------------------------------------------------------------------------



HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20261105_144032_e8888171", "answer_preview": "The base Transformer models were trained on **eight NVIDIA\u202fP100 GPUs** for a total of **12\u202fhours** (\u2248\u202f100,000 training steps, ~0.4\u202fs per step) (nips\u20112", "timestamp": "2026-11-05T09:12:19.913504Z", "level": "info", "event": "Chain invoked successfully"}


Q8: What hardware was used to train the base Transformer models, and for how long?
Expected: They were trained on one machine with 8 NVIDIA P100 GPUs; each base model trained for 100,000 steps, about 12 hours.
Actual:   The base Transformer models were trained on **eight NVIDIA P100 GPUs** for a total of **12 hours** (≈ 100,000 training steps, ~0.4 s per step) (nips‑2017‑attention‑is‑all‑you‑need‑paper_d94cd3.pdf, p. 2; nips‑2017‑attention‑is‑all‑you‑need‑paper_d94cd3.pdf, p. 7).

--------------------------------------------------------------------------------



In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
 
try:
    from langsmith import evaluate  
except ImportError:
    from langsmith.evaluation import evaluate  
 

judge_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_retries=6)
 
judge_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a teacher grading a quiz. You are given a QUESTION, the TRUE ANSWER and the STUDENT ANSWER.\n"
        "Grade the student answer only on factual accuracy compared with the true answer.\n"
        "- CORRECT: it states the same key facts as the true answer, even in different words, "
        "and does not contradict the true answer.\n"
        "- INCORRECT: it misses a key fact, contradicts the true answer, or says the information was not found.\n"
        "Ignore style, length and citations such as (file.pdf, p. 3).\n"
        "Write one or two sentences of reasoning, then end with a final line that is exactly "
        "GRADE: CORRECT or GRADE: INCORRECT."
    )),
    ("human", "QUESTION: {question}\n\nTRUE ANSWER: {reference}\n\nSTUDENT ANSWER: {prediction}"),
])
judge_chain = judge_prompt | judge_llm
 
 
def qa_correctness(run, example) -> dict:
    """LangSmith evaluator: score 1 (correct) or 0 (incorrect), plus the judge's reasoning."""
    verdict = judge_chain.invoke({
        "question": example.inputs["question"],
        "reference": example.outputs["answer"],
        "prediction": (run.outputs or {}).get("answer", ""),
    }).content
 
    grade_lines = [line for line in verdict.splitlines() if "GRADE" in line.upper()]
    grade = grade_lines[-1].upper() if grade_lines else ""
    score = 1 if ("CORRECT" in grade and "INCORRECT" not in grade) else 0
    return {"key": "qa_correctness", "score": score, "comment": verdict.strip()}

cfg = get_rag().cfg
active_llm = cfg.get("active_llm", "groq")
 
experiment_results = evaluate(
    answer_attention_question,
    data=dataset_name,
    evaluators=[qa_correctness],
    experiment_prefix="attention-paper-qa-rag",
    max_concurrency=1, 
    metadata={
        "variant": "RAG with FAISS, local embeddings and Groq",
        "paper": "Attention Is All You Need",
        "embedding_model": cfg["embedding_model"]["model_name"],
        "llm": cfg["llm"][active_llm]["model_name"],
        "chunk_size": cfg.get("chunking", {}).get("chunk_size"),
        "chunk_overlap": cfg.get("chunking", {}).get("chunk_overlap"),
        "top_k": cfg.get("retriever", {}).get("top_k"),
        "search_type": cfg.get("retriever", {}).get("search_type"),
    },
)
 
print("\nEvaluation completed! Open the experiment in LangSmith for per-question results.")
 

results_df = experiment_results.to_pandas()
results_df

View the evaluation results for experiment: 'attention-paper-qa-rag-c6f8c5f5' at:
https://smith.langchain.com/o/b079cf00-4d04-496a-8ed2-8785686b532f/datasets/d55d9379-fed8-41c5-9327-e4565833e659/compare?selectedSessions=f25fccfd-4591-4f1d-a62d-cf901d2c0b66




0it [00:00, ?it/s]HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20261105_144032_e8888171", "answer_preview": "The base Transformer encoder stacks **6 identical layers** (N\u202f=\u202f6) (nips\u20112017\u2011attention\u2011is\u2011all\u2011you\u2011need\u2011paper_d94cd3.pdf, p.\u202f2).", "timestamp": "2026-11-05T09:12:44.202947Z", "level": "info", "event": "Chain invoked successfully"}
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
Retrying request to /openai/v1/chat/completions in 7.000000 seconds
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20261105_144032_e8888171", "answer_preview": "The Transformer injects word\u2011order information by adding **positional encodings** to the input embeddings, giving the model a sense o


Evaluation completed! Open the experiment in LangSmith for per-question results.


,inputs.question,outputs.answer,error,reference.answer,feedback.qa_correctness,execution_time,example_id,id
0,How many identical layers are stacked in the e...,The base Transformer encoder stacks **6 identi...,None,The encoder is a stack of N = 6 identical layers.,1,0.621182,04da72cb-b01c-465c-851f-af6854452697,01a1aafb-613d-7439-a6d7-aca3b8946991
1,How does the Transformer give the model inform...,The Transformer injects word‑order information...,None,It adds positional encodings to the input embe...,1,8.754217,0bac070d-6e71-4430-a8c6-760cdc10a4eb,01a1aafb-63aa-744e-aac9-7e2edcb78498
2,What BLEU score did the big Transformer achiev...,The big Transformer achieved a BLEU score of *...,None,"The big Transformer achieved 28.4 BLEU, improv...",1,19.926881,64e8ee01-cfc2-44df-a3a3-340aa6886d53,01a1aafb-85de-7b26-b635-5abe0ab739e0
3,What is the maximum path length between any tw...,The maximum path length between any two positi...,None,In a self-attention layer any two positions ar...,1,11.751899,67b82ca0-19d8-4cb5-82b3-b89330dbeef0,01a1aafb-d3b5-74de-a45f-aad159d6ccb7
4,How many parallel attention heads does the bas...,The base Transformer uses **8 parallel attenti...,None,The base model uses h = 8 parallel attention h...,1,16.346221,7200e0ba-9900-4c71-b380-69ec494ad584,01a1aafc-019d-7fe6-9ce5-6baa377c217b
5,What hardware was used to train the base Trans...,The base Transformer models were trained on **...,None,They were trained on one machine with 8 NVIDIA...,1,16.424771,a6cfeda3-291b-4225-969b-8bd210eb441e,01a1aafc-4177-7a31-b5df-96d0b492e126
6,Which optimizer and learning-rate schedule wer...,The Transformer was trained with the **Adam op...,None,"The Adam optimizer with beta1 = 0.9, beta2 = 0...",1,15.969640,d537014f-f083-49ed-a51c-db5d68936888,01a1aafc-81a0-7a92-9904-5f6f18721599
7,Why are the dot products scaled by 1/sqrt(d_k)...,The dot products are scaled by \(1/\sqrt{d_k}\...,None,For large values of d_k the dot products grow ...,1,16.650971,dbfb53f9-76fe-4a27-8e49-a686347f1c25,01a1aafc-c002-7519-a249-737d946ab25e
